# EDA and Hypothesis Testing

This notebook performs exploratory data analysis and hypothesis testing on the MMR rental market data.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import mannwhitneyu, kruskal
import statsmodels.api as sm
from statsmodels.formula.api import ols
import warnings
warnings.filterwarnings('ignore')

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# Load data
df = pd.read_csv('../data/processed/rentals_enriched.csv')
print(f"Loaded {len(df)} rows")
print(f"Columns: {len(df.columns)}")

## 1. Distributions and Correlations

In [ ]:
# Rent distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(df['monthly_rent_inr'], kde=True, ax=axes[0])
axes[0].set_title('Distribution of Monthly Rent')
axes[0].set_xlabel('Monthly Rent (INR)')

sns.histplot(np.log1p(df['monthly_rent_inr']), kde=True, ax=axes[1])
axes[1].set_title('Distribution of Log(Monthly Rent)')
axes[1].set_xlabel('Log(Monthly Rent)')

plt.tight_layout()
plt.savefig('../docs/rent_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Rent - Mean: ₹{df['monthly_rent_inr'].mean():,.0f}, Median: ₹{df['monthly_rent_inr'].median():,.0f}")
print(f"Rent/sqft - Mean: ₹{df['rent_per_sqft_carpet_inr'].mean():.2f}, Median: ₹{df['rent_per_sqft_carpet_inr'].median():.2f}")

In [ ]:
# Correlation matrix
numeric_cols = ['monthly_rent_inr', 'rent_per_sqft_carpet_inr', 'carpet_area_sqft', 
                'bhk', 'nearest_station_distance_km', 'min_dist_to_business_hub_km']
corr_matrix = df[numeric_cols].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', center=0, fmt='.2f')
plt.title('Correlation Matrix')
plt.tight_layout()
plt.savefig('../docs/correlation_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

print("Key correlations with rent_per_sqft:")
print(corr_matrix['rent_per_sqft_carpet_inr'].sort_values(ascending=False))

In [ ]:
# Rent/sqft by region and BHK
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# By region
sns.boxplot(data=df, x='region', y='rent_per_sqft_carpet_inr', ax=axes[0])
axes[0].set_title('Rent/sqft by Region')
axes[0].set_ylabel('Rent/sqft (INR)')
axes[0].tick_params(axis='x', rotation=45)

# By BHK
sns.boxplot(data=df, x='bhk', y='rent_per_sqft_carpet_inr', ax=axes[1])
axes[1].set_title('Rent/sqft by BHK')
axes[1].set_ylabel('Rent/sqft (INR)')

plt.tight_layout()
plt.savefig('../docs/rent_by_region_bhk.png', dpi=150, bbox_inches='tight')
plt.show()

## 2. Hypothesis Testing

### H1: Furnished flats have higher rent/sqft than unfurnished

In [ ]:
# Test H1: Furnished vs Unfurnished
furnished = df[df['furnishing'] == 'fully_furnished']['rent_per_sqft_carpet_inr']
unfurnished = df[df['furnishing'] == 'unfurnished']['rent_per_sqft_carpet_inr']

print(f"Furnished - n={len(furnished)}, Mean: ₹{furnished.mean():.2f}, Median: ₹{furnished.median():.2f}")
print(f"Unfurnished - n={len(unfurnished)}, Mean: ₹{unfurnished.mean():.2f}, Median: ₹{unfurnished.median():.2f}")
print(f"Difference: ₹{furnished.mean() - unfurnished.mean():.2f} ({(furnished.mean()/unfurnished.mean()-1)*100:.1f}%)")

# Mann-Whitney U test (non-parametric)
stat, p_value = mannwhitneyu(furnished, unfurnished, alternative='greater')
print(f"\nMann-Whitney U test: statistic={stat:.2f}, p-value={p_value:.2e}")

# Effect size (Cohen's d)
pooled_std = np.sqrt((furnished.std()**2 + unfurnished.std()**2) / 2)
cohens_d = (furnished.mean() - unfurnished.mean()) / pooled_std
print(f"Cohen's d: {cohens_d:.3f}")

# Bootstrap CI for mean difference
n_bootstraps = 10000
boot_diffs = []
for _ in range(n_bootstraps):
    sample_f = np.random.choice(furnished, size=len(furnished), replace=True)
    sample_u = np.random.choice(unfurnished, size=len(unfurnished), replace=True)
    boot_diffs.append(sample_f.mean() - sample_u.mean())

ci_lower = np.percentile(boot_diffs, 2.5)
ci_upper = np.percentile(boot_diffs, 97.5)
print(f"Bootstrap 95% CI for difference: [₹{ci_lower:.2f}, ₹{ci_upper:.2f}]")

# OLS regression controlling for BHK and locality
print("\nOLS Regression (controlling for BHK and region):")
model_h1 = ols('rent_per_sqft_carpet_inr ~ C(furnishing) + C(bhk) + C(region)', data=df).fit()
print(model_h1.summary())

### H2: Rent/sqft differs between listings within 1km of station vs beyond 2km

In [ ]:
# Test H2: Station proximity impact
near_station = df[df['near_station_1km'] == True]['rent_per_sqft_carpet_inr']
far_station = df[df['nearest_station_distance_km'] > 2]['rent_per_sqft_carpet_inr']

print(f"Within 1km - n={len(near_station)}, Mean: ₹{near_station.mean():.2f}, Median: ₹{near_station.median():.2f}")
print(f"Beyond 2km - n={len(far_station)}, Mean: ₹{far_station.mean():.2f}, Median: ₹{far_station.median():.2f}")
print(f"Difference: ₹{near_station.mean() - far_station.mean():.2f} ({(near_station.mean()/far_station.mean()-1)*100:.1f}%)")

# Mann-Whitney U test
stat, p_value = mannwhitneyu(near_station, far_station, alternative='greater')
print(f"\nMann-Whitney U test: statistic={stat:.2f}, p-value={p_value:.2e}")

# Effect size
pooled_std = np.sqrt((near_station.std()**2 + far_station.std()**2) / 2)
cohens_d = (near_station.mean() - far_station.mean()) / pooled_std
print(f"Cohen's d: {cohens_d:.3f}")

# Bootstrap CI
boot_diffs = []
for _ in range(n_bootstraps):
    sample_near = np.random.choice(near_station, size=len(near_station), replace=True)
    sample_far = np.random.choice(far_station, size=len(far_station), replace=True)
    boot_diffs.append(sample_near.mean() - sample_far.mean())

ci_lower = np.percentile(boot_diffs, 2.5)
ci_upper = np.percentile(boot_diffs, 97.5)
print(f"Bootstrap 95% CI for difference: [₹{ci_lower:.2f}, ₹{ci_upper:.2f}]")

# OLS regression controlling for size, BHK, locality
print("\nOLS Regression (controlling for area, BHK, and region):")
model_h2 = ols('rent_per_sqft_carpet_inr ~ near_station_1km + carpet_area_sqft + C(bhk) + C(region)', data=df).fit()
print(model_h2.summary())

### H3: Rent/sqft differs significantly across localities

In [ ]:
# Test H3: Localities have different rent/sqft
# Kruskal-Wallis test (non-parametric ANOVA)
locality_groups = [group['rent_per_sqft_carpet_inr'].values for name, group in df.groupby('locality')]

stat, p_value = kruskal(*locality_groups)
print(f"Kruskal-Wallis test: statistic={stat:.2f}, p-value={p_value:.2e}")
print(f"Number of localities: {len(locality_groups)}")

# Calculate effect size (eta-squared approximation)
n_total = len(df)
eta_squared = (stat - len(locality_groups) + 1) / (n_total)
print(f"Eta-squared (effect size): {eta_squared:.3f}")

# Top 5 and bottom 5 localities by rent/sqft
locality_stats = df.groupby('locality')['rent_per_sqft_carpet_inr'].agg(['mean', 'count']).sort_values('mean', ascending=False)
print("\nTop 5 localities by rent/sqft:")
print(locality_stats.head(10))
print("\nBottom 5 localities by rent/sqft:")
print(locality_stats.tail(10))

## 3. Summary of Findings

In [ ]:
print("="*60)
print("SUMMARY OF HYPOTHESIS TEST RESULTS")
print("="*60)

print("\nH1: Furnished flats have higher rent/sqft than unfurnished")
print(f"- Furnished avg: ₹{furnished.mean():.2f}/sqft")
print(f"- Unfurnished avg: ₹{unfurnished.mean():.2f}/sqft")
print(f"- Difference: ₹{furnished.mean() - unfurnished.mean():.2f}/sqft ({(furnished.mean()/unfurnished.mean()-1)*100:.1f}% premium)")
print(f"- Mann-Whitney p-value: {p_value:.2e}")
print(f"- Cohen's d: {cohens_d:.3f}")
print(f"- Bootstrap 95% CI: [₹{ci_lower:.2f}, ₹{ci_upper:.2f}]")
print(f"- CONCLUSION: {'REJECTED' if p_value < 0.05 else 'FAILED TO REJECT'}")

print("\nH2: Rent/sqft differs by station proximity")
print(f"- Within 1km avg: ₹{near_station.mean():.2f}/sqft")
print(f"- Beyond 2km avg: ₹{far_station.mean():.2f}/sqft")
print(f"- Difference: ₹{near_station.mean() - far_station.mean():.2f}/sqft ({(near_station.mean()/far_station.mean()-1)*100:.1f}% premium)")
print(f"- Mann-Whitney p-value: {p_value:.2e}")
print(f"- Cohen's d: {cohens_d:.3f}")
print(f"- CONCLUSION: {'REJECTED' if p_value < 0.05 else 'FAILED TO REJECT'}")

print("\nH3: Rent/sqft differs across localities")
print(f"- Kruskal-Wallis p-value: {p_value:.2e}")
print(f"- Eta-squared: {eta_squared:.3f}")
print(f"- CONCLUSION: {'REJECTED' if p_value < 0.05 else 'FAILED TO REJECT'}")